# HoloForge Part 2 -- Colab GPU manifest runner

Runs `experiments/run_manifest.py` manifests (E1-E6) on a Colab T4 GPU, writing results directly to a Google-Drive-backed directory so nothing is lost if the session dies mid-run.

**Every cell in this notebook is idempotent** -- re-running the whole notebook top to bottom after a session death (or just re-running a cell) does nothing harmful: cloning skips if the repo already exists (pulls instead), and `run_manifest.py`'s resume logic skips any job whose result file already exists on Drive.

**Before running anything**: Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4 is enough; see the repo's PR discussion for why -- this workload is overhead-bound, not FLOP-bound, at this problem size).

## 1. Health check
Run this FIRST. If `torch.cuda.is_available()` is False, stop and fix the runtime type before continuing -- every later cell assumes a real GPU and will refuse to silently fall back to CPU (`run_manifest.py`'s `get_device()` prints a loud warning rather than pretending).

In [ ]:
!nvidia-smi
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU -- fix Runtime > Change runtime type before continuing."

## 2. Mount Google Drive
Results are written directly under `/content/drive/MyDrive/holoforge_results/` -- a real Drive path, not Colab's ephemeral local disk, so every atomic per-job write already IS the backup (stronger than periodic rsync, which can lose the in-flight job if the runtime is reclaimed mid-copy).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RESULTS_DIR = '/content/drive/MyDrive/holoforge_results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Results will be written to:", RESULTS_DIR)

## 3. Clone or pull the repo
Idempotent: if `/content/HoloForge` already exists (e.g. this cell already ran once this session, or a prior session left it), pulls instead of re-cloning.

In [ ]:
import os
REPO_DIR = '/content/HoloForge'
BRANCH = 'media-part2-followups'

if os.path.isdir(REPO_DIR):
    print("repo already present, pulling latest...")
    !cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}
else:
    !git clone -b {BRANCH} https://github.com/ultramagnus23/HoloForge.git {REPO_DIR}

%cd {REPO_DIR}/holography_media
!git rev-parse HEAD

## 4. Install dependencies
Colab's GPU image already ships a CUDA-enabled torch build -- this just confirms `torch`/`numpy` are present, and installs `torcwa` for the (separate, non-manifest) RCWA cross-check script if you plan to run E7.

In [ ]:
!pip install -q torch numpy
!pip install -q torcwa || echo "torcwa install failed -- fine unless you're running rcwa_crosscheck.py/E7"

## 5. Repo/commit sanity check
Prints the commit this notebook is about to run against -- cross-check it against what you expect (e.g. the PR/branch you're testing) before spending GPU time. This notebook does not hard-pin a commit (it always runs the branch's current HEAD), since pinning would make it silently stale as the branch moves.

In [ ]:
!git log -1 --oneline
!python -c "import sys; sys.path.insert(0,'.'); from experiments.manifest import BUILDERS; [print(k, len(v())) for k,v in BUILDERS.items()]"

## 6. Probe mode -- run this BEFORE any full manifest
One representative job per experiment at full scale, extrapolated to a total T4-hour estimate. Prints the Gate-1 check (40h threshold per the master prompt) but does **not** decide anything -- if it's over budget, report the printed table back and pick a reduction option before running Step 7.

In [ ]:
!python -m experiments.run_manifest --manifest all --probe

## 7. Run a manifest
`--max-minutes` should be comfortably under Colab's session limit (free tier: ~90min idle / up to ~12h wall-clock, but budget conservatively -- 170 is a reasonable default for a ~3h interactive session). The runner exits cleanly before this limit, finishing whatever job is in flight and never truncating one. Re-run this exact cell to resume after a disconnect -- already-done jobs (found on Drive) are skipped automatically.

Change `--manifest` to `E1`, `E2`, ..., `E6`, or `all` as needed. Run experiments in priority order (E1 first -- it's the headline result).

In [ ]:
MANIFEST = 'E1'  # change to E2/E3/E4/E5/E6/all
MAX_MINUTES = 170

!python -m experiments.run_manifest --manifest {MANIFEST} --max-minutes {MAX_MINUTES} --results-dir {RESULTS_DIR}

## 8. Re-run cell 7 to resume
If the cell above exited due to `--max-minutes`, or the runtime disconnected mid-run, just re-run cell 7 (same `MANIFEST` value) -- it picks up exactly where it left off. Repeat until `run_manifest` prints `manifest '<name>' complete: 0 run this session, N already done, N total.`

## 9. When a manifest finishes: copy results back into the git repo
Results live on Drive during the run (survives session death); once a manifest is fully done, copy the relevant subtree into the repo's `results/` directory and commit from your local machine (this notebook does not push -- committing GPU-generated results is a decision for you to review first, not something to automate here).

In [ ]:
import shutil
src = os.path.join(RESULTS_DIR, MANIFEST)
dst = os.path.join(REPO_DIR, 'holography_media', 'results', MANIFEST)
if os.path.isdir(src):
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"copied {src} -> {dst}")
    print("Now zip and download holography_media/results/, or push from Colab if you've set up git credentials:")
    print(f"  !cd {REPO_DIR} && git add holography_media/results/{MANIFEST} && git commit -m '...' && git push")
else:
    print(f"no results found at {src} yet")